# IIIF in Python (no local copies): fetch, preview, and analyze images on-the-fly

This notebook demonstrates a **research-workflow-friendly** pattern for working with image datasets exposed via **IIIF Presentation API v3** (manifests) and the **IIIF Image API** (derivatives, regions, sizes).

Key ideas:

- Use the **manifest** to discover canvases (pages/frames) and their associated **Image API service**.
- Request **only the pixels you need** (thumbnail, region, or downsampled version) using Image API URL parameters.
- Perform lightweight imaging analysis in-memory (NumPy arrays) without downloading full-resolution originals.

> Example data source in this notebook: a IIIF v3 manifest served by 4TU.ResearchData.


In [ ]:
# If you run this notebook on a fresh environment (e.g., Binder), you may need:
# !pip install requests pillow matplotlib numpy

import json
from dataclasses import dataclass
from io import BytesIO
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image

plt.rcParams["figure.dpi"] = 120


## 1) Load a IIIF v3 manifest

A manifest is the entry point for a dataset of images in IIIF Presentation API.  
From it, we can discover canvases and locate the Image API service for each image.


In [ ]:
MANIFEST_URL = "https://data.4tu.nl/iiif/v3/bcf01712-4f8d-4f12-bfc6-84fed4ddc086/1/manifest"

manifest: Dict[str, Any] = requests.get(MANIFEST_URL, timeout=30).json()

print("Manifest type:", manifest.get("type"))
print("Label:", manifest.get("label"))
print("Number of canvases:", len(manifest.get("items", [])))


## 2) Helper utilities

IIIF manifests can vary slightly across servers (e.g., service is a dict vs list; `id` vs `@id`).  
These helpers aim to be robust across common patterns.


In [ ]:
def _first(x: Any, default=None):
    if x is None:
        return default
    if isinstance(x, list):
        return x[0] if x else default
    return x


def get_canvas_label(canvas: Dict[str, Any]) -> str:
    """Best-effort human-readable label for a canvas."""
    label = canvas.get("label")
    if isinstance(label, dict):  # IIIF v3 language map
        # prefer 'en' if present, otherwise first available
        if "en" in label and label["en"]:
            return str(label["en"][0])
        for _, vals in label.items():
            if vals:
                return str(vals[0])
        return ""
    return str(label) if label is not None else ""


def get_image_service_id_from_canvas(canvas: Dict[str, Any]) -> Optional[str]:
    """Extract the IIIF Image API service base URL from a IIIF v3 canvas."""
    # Typical v3 structure:
    # canvas['items'][0] -> annotationPage
    # annotationPage['items'][0] -> annotation
    # annotation['body'] -> image resource
    # body['service'] -> image service (list or dict)
    ann_page = _first(canvas.get("items"), default={})
    ann = _first(ann_page.get("items"), default={})
    body = ann.get("body") or {}

    service = body.get("service")
    service = _first(service, default=None)
    if not isinstance(service, dict):
        return None

    # IIIF Image API v3 uses 'id'; v2 often used '@id'
    return service.get("id") or service.get("@id")


def build_iiif_image_url(
    service_id: str,
    region: str = "full",
    size: str = "!300,300",
    rotation: str = "0",
    quality: str = "default",
    fmt: str = "jpg",
) -> str:
    """Construct an Image API URL (works for many v2/v3 servers)."""
    service_id = service_id.rstrip("/")
    return f"{service_id}/{region}/{size}/{rotation}/{quality}.{fmt}"


def fetch_pil_image(url: str, timeout: int = 60) -> Image.Image:
    """Fetch an image URL into memory as a PIL Image (no local file written)."""
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")


## 3) Preview a few thumbnails

We request small thumbnails to keep bandwidth and memory usage low.


In [ ]:
canvases: List[Dict[str, Any]] = manifest.get("items", [])

thumbs: List[Tuple[str, Image.Image]] = []
for i, canvas in enumerate(canvases[:8]):  # limit for display
    label = get_canvas_label(canvas) or f"Canvas {i}"
    service_id = get_image_service_id_from_canvas(canvas)
    if not service_id:
        print(f"Skipping {label}: no image service found")
        continue

    thumb_url = build_iiif_image_url(service_id, region="full", size="!300,300")
    img = fetch_pil_image(thumb_url)
    thumbs.append((label, img))

len(thumbs), [t[0] for t in thumbs[:3]]


In [ ]:
# Display thumbnails in a small grid
n = len(thumbs)
cols = 4
rows = int(np.ceil(n / cols))

plt.figure(figsize=(10, 2.5 * rows))
for idx, (label, img) in enumerate(thumbs, start=1):
    ax = plt.subplot(rows, cols, idx)
    ax.imshow(img)
    ax.set_title(label, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 4) Lightweight image analysis (in-memory)

Example: compute a simple *brightness* metric per canvas using only thumbnails:

- convert to grayscale (luminance)
- compute mean pixel intensity
- plot the result

This pattern scales: you can swap the metric for whatever your workflow needs (focus score, texture, segmentation, QC checks, etc.).


In [ ]:
def brightness_score(img: Image.Image) -> float:
    arr = np.asarray(img).astype(np.float32)
    # Convert RGB to luma (Rec. 601)
    y = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
    return float(y.mean())


scores = []
labels = []

for i, canvas in enumerate(canvases):
    label = get_canvas_label(canvas) or f"Canvas {i}"
    service_id = get_image_service_id_from_canvas(canvas)
    if not service_id:
        continue

    # Use a small, fixed-size derivative for consistent scoring
    url = build_iiif_image_url(service_id, size="!256,256")
    img = fetch_pil_image(url)
    labels.append(label)
    scores.append(brightness_score(img))

print("Computed scores for", len(scores), "canvases")


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(scores, marker="o")
plt.title("Mean brightness (thumbnail-derived)")
plt.xlabel("Canvas index")
plt.ylabel("Mean luma (0–255)")
plt.grid(True, alpha=0.3)
plt.show()


## 5) Requesting *regions* (avoid full-frame downloads)

The Image API lets you request only a subset of pixels.

Common approaches:

- `region="pct:x,y,w,h"` to request a percentage of the image area (works well when pixel dimensions vary)
- `size="!512,512"` to downsample the region for analysis

Below, we fetch the *central* region of the first available canvas and run a simple edge magnitude computation.


In [ ]:
def sobel_edge_magnitude(gray: np.ndarray) -> np.ndarray:
    """Small Sobel implementation using NumPy only (no SciPy dependency)."""
    # Sobel kernels
    kx = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]], dtype=np.float32)
    ky = np.array([[-1, -2, -1],
                   [ 0,  0,  0],
                   [ 1,  2,  1]], dtype=np.float32)

    # Pad and convolve (naive but fine for small thumbnails/regions)
    g = gray.astype(np.float32)
    gp = np.pad(g, 1, mode="edge")

    gx = np.zeros_like(g, dtype=np.float32)
    gy = np.zeros_like(g, dtype=np.float32)

    for i in range(g.shape[0]):
        for j in range(g.shape[1]):
            patch = gp[i:i+3, j:j+3]
            gx[i, j] = np.sum(patch * kx)
            gy[i, j] = np.sum(patch * ky)

    mag = np.sqrt(gx**2 + gy**2)
    return mag


# Pick first canvas that has an image service
first_canvas = next((c for c in canvases if get_image_service_id_from_canvas(c)), None)
service_id = get_image_service_id_from_canvas(first_canvas) if first_canvas else None
assert service_id, "No canvas with an image service found."

# central 50% region (x=25%, y=25%, width=50%, height=50%)
region_url = build_iiif_image_url(service_id, region="pct:25,25,50,50", size="!512,512")
region_img = fetch_pil_image(region_url)

arr = np.asarray(region_img).astype(np.float32)
gray = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
edges = sobel_edge_magnitude(gray)

# Display: region + edges
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(region_img)
plt.title("Requested region (downsampled)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(edges, cmap="gray")
plt.title("Sobel edge magnitude")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Region URL used:", region_url)


## 6) Next steps for real workflows

Ideas you can extend from here:

- Use `info.json` from the Image API service to learn supported sizes/tiles and the full pixel dimensions.
- Use `region` and `size` strategically for QC pipelines (blur detection, over/under-exposure, artifacts).
- For large collections, parallelize thumbnail/region fetching (respecting server rate limits).
- Keep derived measurements + references (canvas `id`, image service `id`) as the “analysis output”, rather than saving images.

If you share your current analysis goals (segmentation? feature extraction? QC metrics?), I can tailor the last sections into a more domain-specific pipeline.
